In [2]:
# Install necessary libraries for building the RAG system:
# faiss-cpu: For efficient similarity search and indexing of embeddings.
# sentence-transformers: For generating sentence embeddings.
# openai: (Originally for OpenAI API, but not used with Gemini setup now, can be removed if not needed for other parts)
pip install faiss-cpu sentence-transformers openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.0 MB/s eta 0:00:00


In [3]:
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "FAISS is a vector database developed by Facebook.",
    "Embeddings convert text into numerical vectors.",
    "LLMs generate text based on input prompts."
]

In [4]:
# Import the SentenceTransformer library to convert text into numerical embeddings.
# 'all-MiniLM-L6-v2' is a pre-trained model suitable for generating embeddings.
from sentence_transformers import SentenceTransformer

# Load the pre-trained sentence embedding model.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all the documents into numerical vector representations.
# These embeddings will be used by FAISS for similarity search.
doc_embeddings = model.encode(documents)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Define a list of sample documents that will form our knowledge base.
# These are simple strings that our RAG system will retrieve from.
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "FAISS is a vector database developed by Facebook.",
    "Embeddings convert text into numerical vectors.",
    "LLMs generate text based on input prompts."
]

In [5]:
# Import the FAISS library for efficient similarity search.
# Import numpy for numerical operations.
import faiss
import numpy as np

# Get the dimension of the embeddings (length of each vector).
dimension = doc_embeddings.shape[1]

# Initialize a FAISS index (FlatL2 is a simple index using L2 distance).
index = faiss.IndexFlatL2(dimension)

# Add the document embeddings to the FAISS index.
# The .add() method expects a numpy array of floats.
index.add(np.array(doc_embeddings))

In [11]:
# Define the user's query.
query = "What is RAG? Eplain in 200 words?"

# Encode the query into a numerical vector using the same SentenceTransformer model.
query_embedding = model.encode([query])

# Set 'k' to the number of top relevant documents to retrieve.
k = 2  # top results

# Perform a similarity search on the FAISS index using the query embedding.
# It returns distances (lower is more similar) and indices of the top-k documents.
distances, indices = index.search(query_embedding, k)

# Retrieve the actual documents using the obtained indices.
retrieved_docs = [documents[i] for i in indices[0]]

# Print the retrieved documents.
print(retrieved_docs)

['RAG stands for Retrieval Augmented Generation.', 'Embeddings convert text into numerical vectors.']


In [12]:
# Import the Google Generative AI library.
from google import genai

# Initialize the Generative AI client with your API key.
# Ensure your Gemini API key is securely stored in Colab secrets under a name like 'GOOGLE_API_KEY'
# and fetched using userdata.get('GOOGLE_API_KEY').
# The current code uses a hardcoded key, which is not recommended for production.
client = genai.Client(
    api_key="AQ.Ab8RN6IGrG2NYsA48wmYSoQZrhUNInqY8ewwBxi1SsOQh-exRA" # Consider using userdata.get('GOOGLE_API_KEY') here
)

# Join the retrieved documents to form the context for the language model.
context = "\n".join(retrieved_docs)

# Construct the RAG prompt, instructing the model to answer based on the provided context only.
prompt = f"""
Answer based on context only.

Context:
{context}

Question:
{query}
"""

# Generate a response using the Gemini model ('gemini-2.5-flash').
# The 'contents' parameter takes the constructed prompt.
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# Print the generated text response.
print(response.text)

RAG stands for Retrieval Augmented Generation. This name precisely defines the nature of RAG as a process that involves three distinct components: Retrieval, Augmentation, and Generation. While the specific operational details of how these components interact or what they individually accomplish are not provided within the given context, the very phrase 'Retrieval Augmented Generation' conveys that RAG is a system or methodology designed to retrieve information, subsequently augment something with that retrieved information, and finally generate an output.

In related concepts, or within systems that might employ RAG, embeddings play a crucial role. Embeddings are defined as the process of converting text into numerical vectors. This conversion is fundamental because it transforms qualitative textual data into a quantitative format. Numerical vectors can be processed by algorithms and computational models more efficiently than raw text. This capability for mathematical representation o